<a href="https://colab.research.google.com/github/FahemAME/Application-Laravel.github.io/blob/main/FeaturesExtractionComputation_D1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
from scipy.fftpack import fft
from scipy import stats
import time
import ast # Import the ast module

# !pip install fastdtw
# !pip install sktime
# @title helper

def calculate_dtw_distance(x, y):
    """Calculate DTW distance between two sequences"""
    from fastdtw import fastdtw

    # Ensure arrays are 1D and convert to lists for fastdtw
    x = np.array(x).flatten().tolist()
    y = np.array(y).flatten().tolist()
    distance, _ = fastdtw(x, y)
    return distance

def calculate_edr_distance(x, y, epsilon=0.1):
    """Calculate EDR distance between two sequences"""
    from sktime.distances import edr_distance
    x = np.array(x).flatten()
    y = np.array(y).flatten()
    # Ensure same length for EDR
    min_len = min(len(x), len(y))
    edr = edr_distance(x[:min_len], y[:min_len], epsilon=epsilon)
    return edr


# def edr_like(x, y, threshold=0.5):
#     n, m = len(x), len(y)
#     dp = np.zeros((n+1, m+1))

#     for i in range(1, n+1):
#         for j in range(1, m+1):
#             cost = 0 if abs(x[i-1] - y[j-1]) < threshold else 1
#             dp[i][j] = min(
#                 dp[i-1][j] + 1,
#                 dp[i][j-1] + 1,
#                 dp[i-1][j-1] + cost
#             )

#     return dp[n][m]

def extract_EDR_DTW_features(WindowX, WindowY, WindowZ, Magnitude, Name="EDR_DTW"):

    # Convert to numpy arrays if they aren't already
    WindowX = np.array(WindowX).flatten()
    WindowY = np.array(WindowY).flatten()
    WindowZ = np.array(WindowZ).flatten()
    Magnitude = np.array(Magnitude).flatten()

    start_time_set = time.time()

    # ========== DTW and EDR with squared signal ==========
    square_signal = np.square(Magnitude)

    # Handle potential length mismatches
    min_len_mag = min(len(Magnitude), len(square_signal))
    dtw_square = calculate_dtw_distance(Magnitude[:min_len_mag], square_signal[:min_len_mag])
    edr_square = edr_like(Magnitude[:min_len_mag], square_signal[:min_len_mag])

    # ========== DTW and EDR with double difference ==========
    # Double difference: second derivative

    double_diff_signal = np.diff(np.diff(Magnitude))
    min_len_diff = min(len(Magnitude), len(double_diff_signal))
    dtw_double_diff = calculate_dtw_distance(Magnitude[:min_len_diff], double_diff_signal[:min_len_diff])
    edr_double_diff = edr_like(Magnitude[:min_len_diff], double_diff_signal[:min_len_diff])

    end_time_set = time.time()
    time_set = end_time_set - start_time_set

    return {
        # DTW and EDR features
        # f"{Name}dtw_square": dtw_square,
        # f"{Name}edr_square": edr_square,
        # f"{Name}dtw_double_diff": dtw_double_diff,
        # f"{Name}edr_double_diff": edr_double_diff,
        f"{Name}time_set": time_set,
    }

def extract_TD_features(WindowX, WindowY, WindowZ, Magnitude, Name="TD"):

    # Convert to numpy arrays if they aren't already
    WindowX = np.array(WindowX).flatten()
    WindowY = np.array(WindowY).flatten()
    WindowZ = np.array(WindowZ).flatten()
    Magnitude = np.array(Magnitude).flatten()

    start_time_set = time.time()
    # ========== Mean ==========
    mean_x = np.mean(WindowX)
    mean_y = np.mean(WindowY)
    mean_z = np.mean(WindowZ)
    avg_mean = np.mean([mean_x, mean_y, mean_z])

    # ========== Standard Deviation ==========
    std_x = np.std(WindowX)
    std_y = np.std(WindowY)
    std_z = np.std(WindowZ)
    avg_std = np.mean([std_x, std_y, std_z])

    # ========== Kurtosis ==========
    kurt_x = stats.kurtosis(WindowX)
    kurt_y = stats.kurtosis(WindowY)
    kurt_z = stats.kurtosis(WindowZ)
    avg_kurt = np.mean([kurt_x, kurt_y, kurt_z])

    # ========== Correlation between axes ==========
    corr_xy = np.corrcoef(WindowX, WindowY)[0, 1]
    corr_xz = np.corrcoef(WindowX, WindowZ)[0, 1]
    corr_yz = np.corrcoef(WindowY, WindowZ)[0, 1]

    end_time_set = time.time()
    time_set = end_time_set - start_time_set

    return {
        # Mean features
        # f"{Name}mean_x": mean_x,
        # f"{Name}mean_y": mean_y,
        # f"{Name}mean_z": mean_z,
        # f"{Name}avg_mean": avg_mean,

        # Standard deviation features
        # f"{Name}std_x": std_x,
        # f"{Name}std_y": std_y,
        # f"{Name}std_z": std_z,
        # f"{Name}avg_std": avg_std,

        # Kurtosis features
        # f"{Name}kurt_x": kurt_x,
        # f"{Name}kurt_y": kurt_y,
        # f"{Name}kurt_z": kurt_z,
        # f"{Name}avg_kurt": avg_kurt,

        # Correlation features
        # f"{Name}corr_xy": corr_xy,
        # f"{Name}corr_xz": corr_xz,
        # f"{Name}corr_yz": corr_yz,
        f"{Name}time_set": time_set,

    }

def extract_timing_for_multiple_runs(df, num_runs=1):
    all_results = []

    print(f"Extracting timing information for {len(df)} rows across {num_runs} runs...")
    for idx, row in df.iterrows():
        window_x = ast.literal_eval(row['windowX0'])
        window_y = ast.literal_eval(row['windowY0'])
        window_z = ast.literal_eval(row['windowZ0'])
        magnitude = ast.literal_eval(row['Magnitude0'])
        row_timing = {'Activity': row['Activity'],'User': row['User']}

        # Run multiple times
        for run in range(1, num_runs + 1):
            # Extract TD timing
            td_result = extract_TD_features(window_x, window_y, window_z, magnitude, Name=f"TD_Run{run}")
            row_timing.update(td_result)
            edr_result = extract_EDR_DTW_features(window_x, window_y, window_z, magnitude, Name=f"EDR-DTW_Run{run}")
            row_timing.update(edr_result)
        all_results.append(row_timing)

        # Progress indicator
        if (idx + 1) % 150 == 0:
            print(f" Processed {idx + 1}/{len(df)} rows...")
    return pd.DataFrame(all_results)

In [ ]:

RUNS=5
df = pd.read_csv('/storage/emulated/0/Download/D1Segments_WPR.csv')
all_timing_features = extract_timing_for_multiple_runs(df, num_runs=RUNS)
all_timing_features.head(2)

run_comparison = {}
for i in range(1, RUNS+1):
    td_col = f'TD_Run{i}time_set'
    edr_dtw_col = f'EDR-DTW_Run{i}time_set'
    if td_col in all_timing_features.columns:
        run_comparison[f'TD_Run{i}_Sum'] = all_timing_features[td_col].sum()
    if edr_dtw_col in all_timing_features.columns:
        run_comparison[f'EDR-DTW_Run{i}_Sum'] = all_timing_features[edr_dtw_col].sum()

print("Run Comparison (Total Time for Feature Extraction):")
for key, value in run_comparison.items():
  if key.startswith('TD'):
    print(f" {key}: {value:.6f} seconds, Average of {(value*1.)/2690:.6f} second per Segment")
print("---"*30)
for key, value in run_comparison.items():
  if key.startswith('EDR-DTW'):
    print(f" {key}: {value:.6f} seconds, Average of {(value*1.)/2690:.6f} second per Segment")